# Capability 14: Standardized formatting, including markdown tables and unit-aware presentation

7/7 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 14`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap14_standardized_formatting.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: multi-KPI question should render as a markdown table

**✅ PASS**

In [ ]:
_ = ask("Give me North America's revenue, volume, and EBITDA margin for Q1 2024", '01')

01
USER: Give me North America's revenue, volume, and EBITDA margin for Q1 2024

[intent=data_query | sub_agents=['structured']]

AGENT: ## North America — Q1 2024

| KPI | Value |
| --- | --- |
| Revenue | **$3,593 million** |
| Volume | **21,353 thousand hL** |
| EBITDA Margin | **31.3%** |

At $3,593m revenue and 21,353 khl volume, that implies roughly **$168 per hL** of revenue in the quarter.

Note: the EBITDA margin figure is flagged as computed in the source data, so it's derived rather than a directly reported line item.

Follow-ups you might find useful:
- How does North America's Q1 2024 revenue compare to the same quarter last year?
- What drove the volume trend — beer vs. beyond-beer brands?

## 02: comparison across zones should render as a markdown table

**✅ PASS**

In [ ]:
_ = ask('Compare revenue between North America and EMEA in 2025', '02')

02
USER: Compare revenue between North America and EMEA in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## Revenue comparison — North America vs. EMEA (2025)

Based on the 2025 quarterly-grain revenue data (summed across the four quarters):

| Zone | Revenue (USD m, 2025) |
| --- | --- |
| North America | 14,208.0 |
| EMEA | 9,502.0 |

**Key takeaways**

- **North America generated more revenue than EMEA in 2025** — 14,208 m USD vs. 9,502 m USD, a difference of **4,706 m USD** in North America's favor.
- North America was therefore the larger of the two zones by revenue in 2025.

**Data notes / caveats**

- These figures cover only the two requested zones; other zones (e.g. Latin America, Asia Pacific) were excluded from the query, so this is not a company-wide revenue total.
- The evidence contains only the annual sums for 2025 — no quarterly split or prior-year comparison — so I can't say whether the gap was stable through the year or how it evolved versus 2024.

**S

## 03: revenue figure should carry its unit (USD million)

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '03')

03
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

## 04: volume figure should carry its unit (thousand hL)

**✅ PASS**

In [ ]:
_ = ask("What was EMEA's volume in Q2 2025?", '04')

04
USER: What was EMEA's volume in Q2 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s volume in Q2 2025 was **24,172.0 thousand hectoliters (hL)**.

If you’d like, I can help pull the year-over-year change or a country/brand breakdown for the same period.

## 05: EBITDA margin should be presented as a percentage

**✅ PASS**

In [ ]:
_ = ask("What was Asia Pacific's EBITDA margin in 2025?", '05')

05
USER: What was Asia Pacific's EBITDA margin in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: Asia Pacific's EBITDA margin for 2025 was **29.9%** (calculated as normalized EBITDA divided by revenue, based on quarterly data) [Structured KPI data].

If useful, I can compare this against other zones or show how the margin trended across quarters in 2025.

## 06: a quarterly trend question should render as a table with one row per quarter

**✅ PASS**

In [ ]:
_ = ask("Show me EMEA's revenue for each quarter of 2025", '06')

06
USER: Show me EMEA's revenue for each quarter of 2025

[intent=data_query | sub_agents=['structured']]

AGENT: Here is EMEA's revenue for each quarter of 2025:

| Quarter | Period Label | Revenue (USD million) |
| --- | --- | --- |
| 1 | Q1 2025 | $1,965 |
| 2 | Q2 2025 | $2,489 |
| 3 | Q3 2025 | $2,524 |
| 4 | Q4 2025 | $2,524 |

EMEA revenue grew through the year, with Q4 matching Q3 at $2,524 million.

Would you like to compare these quarterly figures to 2024, or break down EMEA revenue by country or brand?

## 07: a single, simple figure need not force a table (sanity check formatting isn't over-applied)

**✅ PASS**

In [ ]:
_ = ask("What was North America's net profit in FY2025?", '07')

07
USER: What was North America's net profit in FY2025?

[intent=data_query | sub_agents=['structured']]

AGENT: I wasn't able to find a value for North America's net profit in FY2025. The structured KPI query for `net_profit_usd_m` (zone = North America, year = 2025, annual/quarter IS NULL) returned no data — the result table shows "—" rather than a figure.

I also don't have any internal documents, web results, or code-execution outputs for this question to fall back on, so I can't report a number without inventing one, and I won't do that.

To be clear about what's missing: this could mean the FY2025 figure simply isn't recorded yet, the annual aggregate hasn't been loaded, or the metric is captured under a different name or zone label in the data.

If helpful, natural next steps would be:
- Check whether **FY2024** North America net profit exists, to see if it's a data-coverage gap versus a metric gap.
- Look at **quarterly** North America figures for 2025, in case only annual aggr